# Week 1: Transformer refresher and setup check

**Advanced Biomedical AI (BME 6938), Spring 2027**

This notebook does two jobs:

1. **Setup check.** It confirms that your environment (Google Colab or HiPerGator) can run the course notebooks.
2. **Refresher.** It walks through the four ideas from the Week 1 animation clips, in code: tokenization and embeddings, query-key-value attention, the encoder block, and masking.

Runtime: about 5 minutes on a GPU, about 10 minutes on CPU. No dataset is needed.

**What to submit on Canvas:** the printed output of the setup-check cell (Section 0) and your answers to the three exercises at the end.

## Settings

This is the only cell you should need to change between platforms.

- **Colab:** leave everything as is. Use *Runtime > Change runtime type > T4 GPU*.
- **HiPerGator:** set `CACHE_DIR` to a folder on your group's `/blue` storage so model downloads do not fill your home directory.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"   # small encoder-only transformer (6 blocks, 12 heads, d = 768)
CACHE_DIR = None                         # HiPerGator example: "/blue/<group>/<user>/hf_cache"
SEED = 0

## 0. Setup check

Run this cell and copy its output into your Canvas submission.

In [ ]:
import sys, platform, importlib

def setup_check():
    print("Python      :", sys.version.split()[0])
    print("Platform    :", platform.platform())
    print("Environment :", "Google Colab" if "google.colab" in sys.modules else "not Colab (HiPerGator or local)")
    for pkg in ["torch", "transformers", "numpy", "matplotlib"]:
        try:
            print(f"{pkg:<12}:", importlib.import_module(pkg).__version__)
        except ImportError:
            print(f"{pkg:<12}: NOT INSTALLED  ->  pip install {pkg}")
    import torch
    if torch.cuda.is_available():
        print("GPU         :", torch.cuda.get_device_name(0))
        print("GPU memory  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
    else:
        print("GPU         : none found (this notebook still runs on CPU)")

setup_check()

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
# attn_implementation="eager" makes the model return its attention weights
model = AutoModel.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR, attn_implementation="eager").to(DEVICE).eval()
print("Loaded", MODEL_NAME, "on", DEVICE)

## 1. Tokenization and embeddings

A transformer only computes with numbers. The tokenizer splits text into tokens and maps each token to an integer ID. Each ID then selects one row of the embedding matrix.

In [ ]:
sentence = "My name is John"
enc = tokenizer(sentence, return_tensors="pt").to(DEVICE)
ids = enc["input_ids"][0]
tokens = tokenizer.convert_ids_to_tokens(ids)
for t, i in zip(tokens, ids.tolist()):
    print(f"{t:>8}  ->  {i}")

The tokenizer added two special tokens, `[CLS]` and `[SEP]`. Rare words are split into subword pieces, which is how a fixed vocabulary covers clinical terms it has never seen whole:

In [ ]:
for word in ["electrocardiogram", "hyperkalemia", "sepsis", "name"]:
    print(f"{word:>18}  ->  {tokenizer.tokenize(word)}")

In [ ]:
E = model.get_input_embeddings()            # the embedding matrix: one row per vocabulary entry
print("Embedding matrix shape (vocabulary size, d):", tuple(E.weight.shape))

with torch.no_grad():
    tok_emb = E(ids)                        # row lookup: one vector per token
print("Token embeddings for the sentence:", tuple(tok_emb.shape))

Attention has no built-in sense of word order, so a **positional embedding** is added to each token embedding. The sum is the input to the first encoder block.

In [ ]:
with torch.no_grad():
    x0 = model.embeddings(enc["input_ids"])   # token embeddings + positional embeddings (+ layer norm)
print("Input to the first encoder block:", tuple(x0.shape), "= (batch, tokens, d)")

## 2. Query, key, value: attention by hand

First, the numbers from the clip. The query of "My" is compared with the key of each of the four tokens. Softmax turns the four scores into weights that sum to 1.

In [ ]:
scores = torch.tensor([2.1, 1.4, 1.0, 0.9])      # q . k / sqrt(d) for the keys of: My, name, is, John
weights = F.softmax(scores, dim=-1)
print("attention weights:", [round(w, 2) for w in weights.tolist()], " sum =", round(weights.sum().item(), 2))

Now the general operation:

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d}}\right)V$$

In [ ]:
def attention(Q, K, V, mask=None):
    """Scaled dot-product attention.
    Q, K, V: (..., tokens, d).  mask: boolean (..., tokens, tokens), True where attention is NOT allowed.
    Returns the output and the attention weights."""
    d = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d)
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

n_tokens, d = 4, 256                      # the 4 x 256 example from the slides
X = torch.randn(n_tokens, d)
W_Q, W_K, W_V = (nn.Linear(d, d, bias=False) for _ in range(3))   # three learned projections
with torch.no_grad():
    Q, K, V = W_Q(X), W_K(X), W_V(X)
    out, A = attention(Q, K, V)

print("output shape:", tuple(out.shape), "(same as the input)")
print("each row of the attention matrix sums to 1:", torch.allclose(A.sum(-1), torch.ones(n_tokens)))

# check against the PyTorch built-in
ref = F.scaled_dot_product_attention(Q, K, V)
print("matches torch built-in:", torch.allclose(out, ref, atol=1e-5))

### Attention in a pretrained model

The same matrix, taken from a trained model. DistilBERT has 6 blocks with 12 heads each, so there are 72 attention matrices for one sentence.

In [ ]:
def get_attentions(text):
    enc = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model(**enc, output_attentions=True)
    toks = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    return toks, torch.stack(out.attentions).squeeze(1).cpu(), out.last_hidden_state[0].cpu()   # (layers, heads, T, T)

def plot_attention(toks, att, layer, head, ax=None):
    ax = ax or plt.subplots(figsize=(4.6, 4))[1]
    im = ax.imshow(att[layer, head], cmap="YlOrBr", vmin=0, vmax=1)
    ax.set_xticks(range(len(toks))); ax.set_xticklabels(toks, rotation=60, ha="right")
    ax.set_yticks(range(len(toks))); ax.set_yticklabels(toks)
    ax.set_xlabel("key"); ax.set_ylabel("query"); ax.set_title(f"block {layer + 1}, head {head + 1}")
    return im

text = "The patient denies chest pain but reports shortness of breath."
toks, att, hidden = get_attentions(text)
print("attention tensor:", tuple(att.shape), "= (blocks, heads, tokens, tokens)")
print("rows sum to 1:", torch.allclose(att.sum(-1), torch.ones(att.shape[:-1]), atol=1e-4))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (l, h) in zip(axes, [(0, 0), (2, 5), (5, 3)]):
    plot_attention(toks, att, l, h, ax)
plt.tight_layout(); plt.show()

## 3. The encoder block

One block is: multi-head attention, add and normalize, feed-forward, add and normalize. The output has the same shape as the input, which is why blocks can be stacked.

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d=256, n_heads=4, d_ff=1024, dropout=0.0):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d))
        self.norm2 = nn.LayerNorm(d)

    def forward(self, x, mask=None):
        a, w = self.attn(x, x, x, attn_mask=mask, need_weights=True, average_attn_weights=False)
        x = self.norm1(x + a)                 # skip connection, then normalize
        x = self.norm2(x + self.ff(x))        # feed-forward on each token, skip connection, normalize
        return x, w

x = torch.randn(1, 4, 256)                    # (batch, tokens, d)
block = EncoderBlock()
y, w = block(x)
print("input :", tuple(x.shape))
print("output:", tuple(y.shape))
print("attention weights:", tuple(w.shape), "= (batch, heads, tokens, tokens)")

stack = nn.ModuleList([EncoderBlock() for _ in range(6)])
h = x
for blk in stack:
    h, _ = blk(h)
print("after 6 stacked blocks:", tuple(h.shape))
print("parameters in the 6-block stack:", f"{sum(p.numel() for p in stack.parameters()):,}")

## 4. Masking: what makes a decoder

A decoder predicts the next token, so a token must not attend to later positions. The scores for future positions are set to negative infinity before the softmax, which gives them weight 0.

In [ ]:
T = 3                                                         # Me, llamo, John
causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)   # True above the diagonal = blocked
print("mask (True = blocked):")
print(causal_mask.int())

Xd = torch.randn(T, 64)
_, A_free = attention(Xd, Xd, Xd)
_, A_masked = attention(Xd, Xd, Xd, mask=causal_mask)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
for ax, M, title in zip(axes, [A_free, A_masked], ["no mask (encoder)", "causal mask (decoder)"]):
    ax.imshow(M, cmap="RdPu", vmin=0, vmax=1); ax.set_title(title)
    ax.set_xticks(range(T)); ax.set_xticklabels(["Me", "llamo", "John"]); ax.set_yticks(range(T)); ax.set_yticklabels(["Me", "llamo", "John"])
    for i in range(T):
        for j in range(T):
            ax.text(j, i, f"{M[i, j].item():.2f}", ha="center", va="center")
plt.tight_layout(); plt.show()
print("weights above the diagonal are all zero:", bool((A_masked[causal_mask] == 0).all()))

## Exercises

Answer each in two or three sentences, with the code you used.

**Exercise 1: what does a head attend to?**
Using `get_attentions` and `plot_attention`, find one head in DistilBERT whose pattern you can describe in words (for example: attends to the previous token, to the next token, or to `[SEP]`). Report the block and head number and show the plot for a clinical sentence of your choice.

**Exercise 2: static versus contextual embeddings.**
The word "discharge" means different things in "The patient was ready for discharge home." and "There was purulent discharge from the wound." Compare the cosine similarity of the two "discharge" vectors (a) in the input embedding matrix and (b) in the final hidden state. Explain the difference.

**Exercise 3: attention ignores order.**
Using the `attention` function from Section 2 with `Q = K = V = X`, show that permuting the rows of `X` permutes the rows of the output in the same way. What does this tell you about why positional encodings are needed?

In [ ]:
# Starter for Exercise 2
def word_vectors(text, word):
    """Return (static embedding, contextual embedding) for the first occurrence of a single-token word."""
    toks, _, hidden = get_attentions(text)
    i = toks.index(word)
    wid = tokenizer.convert_tokens_to_ids(word)
    return model.get_input_embeddings().weight[wid].detach().cpu(), hidden[i]

# s1, c1 = word_vectors("The patient was ready for discharge home.", "discharge")
# s2, c2 = word_vectors("There was purulent discharge from the wound.", "discharge")
# print(F.cosine_similarity(s1, s2, dim=0), F.cosine_similarity(c1, c2, dim=0))